# Train v1 Experiment
Run the baseline or curriculum experiment from Databricks using the Python entrypoint.

This notebook installs the required Python packages before importing the training modules so it can be run directly without first opening the separate setup notebook.

In [ ]:
%pip install pyyaml rouge-score mlflow transformers pandas tqdm matplotlib

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%restart_python

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

dbutils.widgets.text("experiment_config", "configs/experiment/curriculum_linear.yaml")
dbutils.widgets.text("runtime_config", "configs/runtime/databricks.yaml")

In [ ]:
import logging

# Suppress noisy Azure/HTTPX logs
for noisy_logger in [
    "azure.core.pipeline.policies.http_logging_policy",
    "httpx",
    "azure.identity",
    "azure.core.pipeline",
]:
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)


In [ ]:
from scripts.train import run_training

experiment_config = dbutils.widgets.get("experiment_config")
runtime_config = dbutils.widgets.get("runtime_config")
result = run_training(experiment_config=experiment_config, runtime_config=runtime_config)
result

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

run_dir = Path(result["run_dir"])
history_path = run_dir / "training_history.json"

if not history_path.exists():
    raise FileNotFoundError(f"Expected training history at: {history_path}")

with history_path.open("r", encoding="utf-8") as handle:
    history_payload = json.load(handle)

history_df = pd.DataFrame(history_payload.get("history", []))
if history_df.empty:
    raise ValueError(f"No history rows found in {history_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["val_rouge1"], marker="o", label="val_rouge1")
axes[1].plot(history_df["epoch"], history_df["val_rougeL"], marker="o", label="val_rougeL")
axes[1].set_title("Validation ROUGE by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
display(history_df)

In [ ]:
from src.curriculum.schedules import build_schedule
from src.utils.io import deep_merge, load_experiment_config, load_yaml

config = load_experiment_config(experiment_config)
if runtime_config:
    config = deep_merge(config, load_yaml(runtime_config))

steps_per_epoch = max(1, len(history_df))  # one epoch-average p_ar point per epoch
curriculum_cfg = config.get("curriculum", {})
schedule_name = curriculum_cfg.get("schedule_type", "unknown")
schedule = build_schedule(config)

expected_p_ar = []
for epoch_idx in range(1, len(history_df) + 1):
    # Approximate epoch midpoint in global-step space for visualization.
    approx_step = int((epoch_idx - 0.5) * steps_per_epoch)
    expected_p_ar.append(float(schedule.get_p_ar(approx_step)))

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["avg_p_ar"], marker="o", label="observed avg_p_ar")
plt.plot(history_df["epoch"], expected_p_ar, marker="o", linestyle="--", label=f"expected {schedule_name} p_ar")
plt.plot(history_df["epoch"], history_df["avg_replacement_rate"], marker="o", label="observed avg_replacement_rate")
plt.title("Curriculum Schedule Progress")
plt.xlabel("Epoch")
plt.ylabel("Rate")
plt.ylim(0.0, 1.0)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

display(
    history_df[["epoch", "avg_p_ar", "avg_replacement_rate"]].assign(
        expected_p_ar=expected_p_ar
    )
)